In [35]:
#!pip install pydantic

In [36]:
# ============================================================
# IMPORTS & SETUP
# ============================================================

import os
import json
import time
import base64
import io
import requests
from pathlib import Path
from typing import List
from datetime import datetime

import pandas as pd
from PIL import Image
from pydantic import BaseModel
from dotenv import load_dotenv
from datasets import load_dataset
from openai import OpenAI

# ── Load API key and initialize client ────────────────────────────
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

if os.getenv("OPENAI_API_KEY"):
    print("✓ API key loaded successfully")
else:
    print("✗ API key not found — check your .env file")

# ── Load dataset ──────────────────────────────────────────────────
print("\nLoading product dataset...")
try:
    dataset = load_dataset(
        "ashraq/fashion-product-images-small", 
        split="train[:100]"
    )
    print(f"✓ Loaded {len(dataset)} products")
    
    products_df = pd.DataFrame(dataset)
    print(f"  Columns: {products_df.columns.tolist()}")

except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("  Using local placeholder data instead...")
    
    products_df = pd.DataFrame([
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
    ])

# ── Create images directory ───────────────────────────────────────
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print(f"\n✓ Setup complete!")
print(f"  Total products: {len(products_df)}")
print(f"  Images directory: {images_dir.resolve()}")

✓ API key loaded successfully

Loading product dataset...
✓ Loaded 100 products
  Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Setup complete!
  Total products: 100
  Images directory: C:\Users\dbyst\OneDrive\Desktop\Ironhack_labs\Ironhack_Day4\product_images


In [37]:
"""
API with JSON Validation using Pydantic - Complete Solution
Validate JSON input using Pydantic before processing
"""
 
#from pydantic import BaseModel, Field, validator
#from typing import Optional
#import json
 
print("="*50)
print("PYDANTIC BASICS")
print("="*50)
 
class SimpleProduct(BaseModel):
    """A simple product model for validation."""
    name: str
    price: float
    quantity: int = 1  # Default value
    
    @validator('price')
    def price_must_be_positive(cls, v):
        """Validate that price is positive."""
        if v <= 0:
            raise ValueError('Price must be positive')
        return v
    
    @validator('quantity')
    def quantity_must_be_positive(cls, v):
        """Validate that quantity is positive."""
        if v <= 0:
            raise ValueError('Quantity must be positive')
        return v
 
# Test validation
print("\n1. Valid data:")
try:
    product1 = SimpleProduct(name="Widget", price=10.99, quantity=5)
    print(f"  ✓ Valid: {product1.name} - ${product1.price}")
except Exception as e:
    print(f"  ✗ Error: {e}")
 
print("\n2. Invalid data (negative price):")
try:
    product2 = SimpleProduct(name="Widget", price=-10.99)
except Exception as e:
    print(f"  ✗ Validation error (expected): {e}")
 
print("\n✓ Pydantic basics working!")

PYDANTIC BASICS

1. Valid data:
  ✓ Valid: Widget - $10.99

2. Invalid data (negative price):
  ✗ Validation error (expected): 1 validation error for SimpleProduct
price
  Value error, Price must be positive [type=value_error, input_value=-10.99, input_type=float]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error

✓ Pydantic basics working!


C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\461198087.py:20: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator('price')
C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\461198087.py:27: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator('quantity')


In [38]:
# ============================================================
# STEP 2: PRODUCT DATA MODELS FOR LISTING GENERATOR
# ============================================================

from pydantic import BaseModel, Field, validator
from typing import Optional, List
from enum import Enum

# ── Allowed categories ────────────────────────────────────────────
class ProductCategory(str, Enum):
    FASHION     = "Fashion"
    ELECTRONICS = "Electronics"
    HOME        = "Home"
    SPORTS      = "Sports"
    BEAUTY      = "Beauty"
    GENERAL     = "General"

# ── Input model — what goes INTO the listing generator ────────────
class ProductListingRequest(BaseModel):
    """Validates product data before sending to ChatGPT API."""
    
    name: str = Field(
        ...,                          # required
        min_length=2,
        max_length=200,
        description="Product name"
    )
    price: float = Field(
        ...,
        gt=0,                         # greater than 0
        lt=100000,
        description="Product price in USD"
    )
    category: ProductCategory = Field(
        default=ProductCategory.GENERAL,
        description="Product category"
    )
    description: Optional[str] = Field(
        default=None,
        max_length=1000,
        description="Optional existing description"
    )
    additional_info: Optional[str] = Field(
        default=None,
        max_length=500,
        description="Extra details e.g. colour, material, features"
    )
    image_available: bool = Field(
        default=True,
        description="Whether a product image is available"
    )

    @validator('name')
    def name_must_not_be_empty(cls, v):
        if not v.strip():
            raise ValueError('Product name cannot be empty or whitespace')
        return v.strip()

    @validator('price')
    def price_rounded(cls, v):
        """Round price to 2 decimal places."""
        return round(v, 2)

    @validator('additional_info')
    def clean_additional_info(cls, v):
        if v is not None:
            return v.strip()
        return v


# ── Output model — what comes BACK from the API ───────────────────
class GeneratedListing(BaseModel):
    """Validates the JSON response from ChatGPT API."""
    
    title: str = Field(
        ...,
        min_length=5,
        max_length=100,
        description="SEO-friendly product title"
    )
    description: str = Field(
        ...,
        min_length=50,
        max_length=2000,
        description="Full product description"
    )
    features: List[str] = Field(
        ...,
        min_items=1,
        max_items=10,
        description="List of key product features"
    )
    keywords: str = Field(
        ...,
        min_length=5,
        description="Comma-separated SEO keywords"
    )

    @validator('features')
    def features_not_empty(cls, v):
        if not all(f.strip() for f in v):
            raise ValueError('All features must be non-empty strings')
        return [f.strip() for f in v]

    @validator('title')
    def title_strip(cls, v):
        return v.strip()


# ── Test the models ───────────────────────────────────────────────
print("=" * 50)
print("STEP 2: PRODUCT MODEL VALIDATION TESTS")
print("=" * 50)

print("\n1. Valid product request:")
try:
    req = ProductListingRequest(
        name="Premium Wireless Headphones",
        price=79.99,
        category="Electronics",
        additional_info="Noise cancelling, 30-hour battery, black"
    )
    print(f"  ✓ Valid: {req.name} | ${req.price} | {req.category}")
except Exception as e:
    print(f"  ✗ Error: {e}")

print("\n2. Invalid — negative price:")
try:
    req2 = ProductListingRequest(name="Headphones", price=-10)
except Exception as e:
    print(f"  ✗ Caught (expected): price must be > 0")

print("\n3. Invalid — empty name:")
try:
    req3 = ProductListingRequest(name="   ", price=29.99)
except Exception as e:
    print(f"  ✗ Caught (expected): name cannot be empty")

print("\n4. Invalid — price too high:")
try:
    req4 = ProductListingRequest(name="Gold Bar", price=999999)
except Exception as e:
    print(f"  ✗ Caught (expected): price out of range")

print("\n5. Valid generated listing:")
try:
    listing = GeneratedListing(
        title="Premium Noise-Cancelling Wireless Headphones",
        description="Experience crystal-clear audio with these premium "
                    "wireless headphones featuring advanced noise cancellation "
                    "technology and an impressive 30-hour battery life.",
        features=["Active noise cancellation", "30-hour battery",
                  "Bluetooth 5.0", "Foldable design"],
        keywords="wireless headphones, noise cancelling, bluetooth, audio"
    )
    print(f"  ✓ Valid listing: {listing.title}")
except Exception as e:
    print(f"  ✗ Error: {e}")

print("\n✓ Product models working!")

STEP 2: PRODUCT MODEL VALIDATION TESTS

1. Valid product request:
  ✓ Valid: Premium Wireless Headphones | $79.99 | ProductCategory.ELECTRONICS

2. Invalid — negative price:
  ✗ Caught (expected): price must be > 0

3. Invalid — empty name:
  ✗ Caught (expected): name cannot be empty

4. Invalid — price too high:
  ✗ Caught (expected): price out of range

5. Valid generated listing:
  ✓ Valid listing: Premium Noise-Cancelling Wireless Headphones

✓ Product models working!


C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\1962091640.py:53: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator('name')
C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\1962091640.py:59: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  @validator('price')
C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\1962091640.py:64: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should m

In [39]:
# ============================================================
# STEP 3: VALIDATING JSON INPUT FROM FILES
# ============================================================

import json
from pathlib import Path
from pydantic import ValidationError

# ── Create example JSON files ─────────────────────────────────────

# Valid products JSON
valid_products = {
    "products": [
        {
            "name": "Men's Running Shoes",
            "price": 89.99,
            "category": "Sports",
            "additional_info": "Lightweight, breathable mesh, size 42"
        },
        {
            "name": "Floral Summer Dress",
            "price": 45.00,
            "category": "Fashion",
            "additional_info": "Cotton blend, floral print, sizes XS-XL"
        },
        {
            "name": "Leather Wallet",
            "price": 34.99,
            "category": "Fashion",
            "additional_info": "Genuine leather, 8 card slots, brown"
        }
    ]
}

# Invalid products JSON — contains deliberate errors
invalid_products = {
    "products": [
        {
            "name": "",              # empty name
            "price": 29.99,
            "category": "Fashion"
        },
        {
            "name": "Broken Product",
            "price": -15.00,         # negative price
            "category": "Fashion"
        },
        {
            "name": "Another Product",
            "price": 999999,         # price too high
            "category": "Fashion"
        },
        {
            "name": "Missing Price"  # no price field
        }
    ]
}

# Save to files
Path("valid_products.json").write_text(
    json.dumps(valid_products, indent=2))
Path("invalid_products.json").write_text(
    json.dumps(invalid_products, indent=2))

print("✓ Example JSON files created:")
print("  valid_products.json")
print("  invalid_products.json")


# ── Validation function ───────────────────────────────────────────

def validate_products_from_json(filepath: str):
    """
    Load and validate products from a JSON file.
    
    Returns:
    - validated: list of valid ProductListingRequest objects
    - errors: list of dicts describing validation failures
    """
    validated = []
    errors = []

    # Load file
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except FileNotFoundError:
        print(f"✗ File not found: {filepath}")
        return [], []
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON in file: {e}")
        return [], []

    products = data.get("products", [])
    print(f"\nValidating {len(products)} products from {filepath}...")
    print("-" * 40)

    for i, product in enumerate(products):
        try:
            validated_product = ProductListingRequest(**product)
            validated.append(validated_product)
            print(f"  [{i+1}] ✓ Valid: {validated_product.name} "
                  f"| ${validated_product.price}")

        except ValidationError as e:
            # Extract clean error messages
            error_messages = []
            for error in e.errors():
                field = error['loc'][0] if error['loc'] else 'unknown'
                msg = error['msg']
                error_messages.append(f"{field}: {msg}")

            errors.append({
                "index": i + 1,
                "data": product,
                "errors": error_messages
            })
            print(f"  [{i+1}] ✗ Invalid: {product.get('name', 'N/A')}")
            for msg in error_messages:
                print(f"       → {msg}")

    return validated, errors


# ── Run validation on both files ──────────────────────────────────
print("=" * 50)
print("STEP 3: JSON VALIDATION")
print("=" * 50)

# Test valid file
valid_results, valid_errors = validate_products_from_json(
    "valid_products.json")

print(f"\n  Summary — valid_products.json:")
print(f"  ✓ Passed: {len(valid_results)}")
print(f"  ✗ Failed: {len(valid_errors)}")

# Test invalid file
invalid_results, invalid_errors = validate_products_from_json(
    "invalid_products.json")

print(f"\n  Summary — invalid_products.json:")
print(f"  ✓ Passed: {len(invalid_results)}")
print(f"  ✗ Failed: {len(invalid_errors)}")

# Show validated objects ready for API
print(f"\n✓ Products ready for API processing: {len(valid_results)}")
for p in valid_results:
    print(f"  → {p.name} | ${p.price} | {p.category.value}")

✓ Example JSON files created:
  valid_products.json
  invalid_products.json
STEP 3: JSON VALIDATION

Validating 3 products from valid_products.json...
----------------------------------------
  [1] ✓ Valid: Men's Running Shoes | $89.99
  [2] ✓ Valid: Floral Summer Dress | $45.0
  [3] ✓ Valid: Leather Wallet | $34.99

  Summary — valid_products.json:
  ✓ Passed: 3
  ✗ Failed: 0

Validating 4 products from invalid_products.json...
----------------------------------------
  [1] ✗ Invalid: 
       → name: String should have at least 2 characters
  [2] ✗ Invalid: Broken Product
       → price: Input should be greater than 0
  [3] ✗ Invalid: Another Product
       → price: Input should be less than 100000
  [4] ✗ Invalid: Missing Price
       → price: Field required

  Summary — invalid_products.json:
  ✓ Passed: 0
  ✗ Failed: 4

✓ Products ready for API processing: 3
  → Men's Running Shoes | $89.99 | Sports
  → Floral Summer Dress | $45.0 | Fashion
  → Leather Wallet | $34.99 | Fashion


In [40]:
# ============================================================
# STEP 4: CREATING THE PRODUCT LISTING PROMPT
# ============================================================

def create_product_listing_prompt(product_name, price, category, 
                                   additional_info=None):
    """
    Create a prompt for generating product listings.
    
    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information
    
    Returns:
    - Formatted prompt string
    """
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}

Be specific about what you see in the image. Mention colors, materials, 
design elements, and any distinctive features."""

    return prompt


# ── Test prompt creation ──────────────────────────────────────────
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

print("\n" + "=" * 50)
print("PROMPT TEMPLATE")
print("=" * 50)
print(test_prompt)

print("\n" + "=" * 50)
print("PROMPT STATS")
print("=" * 50)
print(f"Total characters: {len(test_prompt)}")
print(f"Estimated tokens: ~{len(test_prompt) // 4}")
print("✓ Prompt template ready")


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}

Be specific about what you see in the image. Mention colors, materials, 
desig

In [ ]:
# ============================================================
# IMAGE ENCODING FUNCTION 
# ============================================================

import base64
import io
from PIL import Image

def encode_image_to_base64(image_source):
    """
    Convert an image to base64 format for API transmission.
    Parameters:
    - image_source: PIL Image object, file path (str/Path), or URL string
    Returns:
    - base64 encoded string
    """
    try:
        # If it's a PIL Image
        if hasattr(image_source, 'save'):
            buffer = io.BytesIO()
            if image_source.mode != 'RGB':
                image_source = image_source.convert('RGB')
            image_source.save(buffer, format='JPEG')
            buffer.seek(0)
            return base64.b64encode(buffer.read()).decode('utf-8')

        # If it's a file path
        elif isinstance(image_source, (str, Path)) and \
             Path(image_source).exists():
            with open(image_source, 'rb') as f:
                return base64.b64encode(f.read()).decode('utf-8')

        else:
            raise ValueError(f"Invalid image source: {type(image_source)}")

    except Exception as e:
        print(f"✗ Error encoding image: {e}")
        return None


def verify_encoding(base64_string):
    """Verify base64 string decodes back to a valid image."""
    try:
        image_data = base64.b64decode(base64_string)
        image = Image.open(io.BytesIO(image_data))
        print(f"✓ Encoding verified — size: {image.size}, mode: {image.mode}")
        return True
    except Exception as e:
        print(f"✗ Verification failed: {e}")
        return False


print("✓ encode_image_to_base64 defined and ready")

✓ encode_image_to_base64 defined and ready


In [44]:
# ============================================================
# STEP 5: CALLING THE CHATGPT API WITH VISION
# ============================================================

import json
import time

def call_chatgpt_vision(image_source, prompt, model="gpt-4o",
                         max_tokens=1000):
    """
    Send image + prompt to ChatGPT Vision API.
    
    Parameters:
    - image_source: PIL Image or base64 string
    - prompt: Text prompt
    - model: OpenAI model to use (must be gpt-4o for vision)
    - max_tokens: Maximum tokens in response
    
    Returns:
    - Raw response text from API
    """
    try:
        # Encode image if it's a PIL Image
        if hasattr(image_source, 'save'):
            base64_image = encode_image_to_base64(image_source)
        else:
            base64_image = image_source

        if not base64_image:
            raise ValueError("Image encoding returned None")

        print(f"  Image encoded: {len(base64_image)} chars")

        # Build and send API request
        # Image block BEFORE text block — correct order for vision models
        response = client.chat.completions.create(
            model="gpt-4o",
            max_tokens=max_tokens,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"   # high = better image analysis
                        }
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ]
            }]
        )

        raw_text = response.choices[0].message.content

        print(f"✓ API call successful")
        print(f"  Tokens used — Input: {response.usage.prompt_tokens}, "
              f"Output: {response.usage.completion_tokens}, "
              f"Total: {response.usage.total_tokens}")

        return raw_text

    except Exception as e:
        print(f"✗ API call failed: {e}")
        return None


def parse_json_response(raw_text):
    """
    Parse JSON from API response, handling common formatting issues.
    Returns parsed dict or fallback dict with raw text.
    """
    # Attempt 1 — direct parse
    try:
        return json.loads(raw_text)
    except json.JSONDecodeError:
        pass

    # Attempt 2 — strip markdown code blocks
    try:
        cleaned = raw_text.strip()
        if cleaned.startswith("```"):
            cleaned = cleaned.split("```")[1]
            if cleaned.startswith("json"):
                cleaned = cleaned[4:]
        return json.loads(cleaned.strip())
    except json.JSONDecodeError:
        pass

    # Attempt 3 — extract between first { and last }
    try:
        start = raw_text.find('{')
        end = raw_text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(raw_text[start:end])
    except json.JSONDecodeError:
        pass

    print("⚠ Could not parse JSON — storing raw response")
    return {"raw_response": raw_text}


# ── Test with one product ─────────────────────────────────────────
print("=" * 50)
print("STEP 5: API TEST WITH SINGLE PRODUCT")
print("=" * 50)

# Get first product from dataset
sample = products_df.iloc[0]

# Extract product info
if 'productDisplayName' in products_df.columns:
    product_name = sample.get('productDisplayName', 'Fashion Product')
    category     = sample.get('masterCategory', 'Fashion')
elif 'name' in products_df.columns:
    product_name = sample['name']
    category     = sample.get('category', 'General')
else:
    product_name = "Sample Product"
    category     = "General"

price = sample.get('price', 29.99)
image = sample.get('image', None)

# Build prompt
prompt = create_product_listing_prompt(
    product_name=product_name,
    price=float(price) if isinstance(price, (int, float)) else 29.99,
    category=category
)

print(f"\nProcessing: {product_name}")
print(f"Category:   {category}")
print(f"Image:      {'Available' if image is not None else 'Not available'}")

# Call API
raw_response = call_chatgpt_vision(image, prompt)

if raw_response:
    print("\n--- RAW RESPONSE (first 500 chars) ---")
    print(raw_response[:500])

    # Parse JSON
    parsed = parse_json_response(raw_response)

    print("\n--- PARSED RESULT ---")
    print(f"Title:       {parsed.get('title', 'N/A')}")
    print(f"Features:    {parsed.get('features', [])[:2]}")
    print(f"Keywords:    {str(parsed.get('keywords', 'N/A'))[:80]}...")
    print(f"Description: {str(parsed.get('description', 'N/A'))[:150]}...")

    # Check if model actually saw the image
    raw_lower = raw_response.lower()
    if any(phrase in raw_lower for phrase in [
        "unable to see", "cannot see", "can't see",
        "no image", "without seeing", "assumptions"
    ]):
        print("\n⚠ WARNING: Model may not have seen the image correctly")
        print("  → Check that encode_image_to_base64 is working")
        print("  → Check image size is not 0 bytes")
    else:
        print("\n✓ Model appears to have analysed the image successfully")

else:
    print("✗ No response received — check API key and internet connection")

STEP 5: API TEST WITH SINGLE PRODUCT

Processing: Turtle Check Men Navy Blue Shirt
Category:   Apparel
Image:      Available
  Image encoded: 2388 chars
✓ API call successful
  Tokens used — Input: 490, Output: 263, Total: 753

--- RAW RESPONSE (first 500 chars) ---
```json
{
    "title": "Stylish Navy Blue Check Shirt for Men - Casual Elegance",
    "description": "Upgrade your wardrobe with the Turtle Check Men Navy Blue Shirt, a perfect blend of style and comfort. Crafted from high-quality, breathable fabric, this versatile shirt is designed to keep you looking sharp in any casual setting. The navy blue color is accentuated by a trendy check pattern, offering a modern twist on a classic design. The shirt features a structured collar and full sleeves, mak

--- PARSED RESULT ---
Title:       Stylish Navy Blue Check Shirt for Men - Casual Elegance
Features:    ['Classic navy blue with a contemporary check pattern', 'Made from breathable, high-quality fabric']
Keywords:    men's shirt, 

In [46]:
# ============================================================
# STEP 6: PROCESSING MULTIPLE PRODUCTS IN BATCH
# ============================================================

import json
import time
from datetime import datetime

def process_single_product(row, index):
    """
    Process one product — validate, call API, parse response.
    
    Parameters:
    - row: DataFrame row (one product)
    - index: product index for tracking
    
    Returns:
    - result dict with status and listing data
    """
    try:
        # ── Extract product info ──────────────────────────────────
        if 'productDisplayName' in row:
            name     = row.get('productDisplayName', f'Product {index}')
            category = row.get('masterCategory', 'Fashion')
        elif 'name' in row:
            name     = row.get('name', f'Product {index}')
            category = row.get('category', 'General')
        else:
            name     = f'Product {index}'
            category = 'General'

        price_raw = row.get('price', 29.99)
        price     = float(price_raw) if isinstance(
                    price_raw, (int, float)) else 29.99
        image     = row.get('image', None)

        # ── Skip if no image ──────────────────────────────────────
        if image is None:
            return {
                "index":  index,
                "name":   name,
                "status": "skipped",
                "reason": "no image available"
            }

        # ── Build prompt ──────────────────────────────────────────
        prompt = create_product_listing_prompt(
            product_name=name,
            price=price,
            category=category
        )

        # ── Call API ──────────────────────────────────────────────
        raw_response = call_chatgpt_vision(image, prompt)

        if not raw_response:
            return {
                "index":  index,
                "name":   name,
                "status": "failed",
                "reason": "API returned empty response"
            }

        # ── Parse response ────────────────────────────────────────
        parsed = parse_json_response(raw_response)

        return {
            "index":        index,
            "name":         name,
            "category":     category,
            "price":        price,
            "status":       "success",
            "listing":      parsed,
            "raw_response": raw_response
        }

    except Exception as e:
        return {
            "index":  index,
            "name":   row.get('productDisplayName',
                      row.get('name', f'Product {index}')),
            "status": "error",
            "reason": str(e)
        }


def process_products_batch(df, max_products=3, delay=1.5,
                            save_path="product_listings.json"):
    """
    Process multiple products and save results after each one.
    
    Parameters:
    - df:           DataFrame with products
    - max_products: max to process (keep low to save API credits)
    - delay:        seconds between API calls to avoid rate limits
    - save_path:    where to save results JSON
    
    Returns:
    - list of result dicts
    """
    results    = []
    successful = 0
    failed     = 0
    skipped    = 0

    total = min(max_products, len(df))

    print("=" * 55)
    print(f"STEP 6: BATCH PROCESSING {total} PRODUCTS")
    print("=" * 55)
    print(f"Delay between calls: {delay}s")
    print(f"Estimated cost:      ~${total * 0.004:.3f}")
    print(f"Save path:           {save_path}\n")

    for i in range(total):
        print(f"[{i+1}/{total}] Processing: "
              f"{df.iloc[i].get('productDisplayName', f'Product {i}')[:50]}")
        print("-" * 40)

        row    = df.iloc[i]
        result = process_single_product(row, i)
        results.append(result)

        # ── Track and print status ────────────────────────────────
        if result['status'] == 'success':
            successful += 1
            listing = result['listing']
            print(f"  ✓ Success")
            print(f"    Title:    {listing.get('title', 'N/A')[:60]}")
            print(f"    Features: "
                  f"{len(listing.get('features', []))} items")
            print(f"    Keywords: "
                  f"{str(listing.get('keywords', ''))[:60]}...")

        elif result['status'] == 'skipped':
            skipped += 1
            print(f"  ⚠ Skipped: {result.get('reason')}")

        else:
            failed += 1
            print(f"  ✗ Failed: {result.get('reason')}")

        # ── Save after every product in case of crash ─────────────
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump({
                "generated_at":    datetime.now().isoformat(),
                "total_processed": i + 1,
                "successful":      successful,
                "failed":          failed,
                "skipped":         skipped,
                "results":         results
            }, f, indent=2, ensure_ascii=False)

        # ── Wait between calls ────────────────────────────────────
        if i < total - 1:
            print(f"\n  Waiting {delay}s before next call...\n")
            time.sleep(delay)

    # ── Final summary ─────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("BATCH COMPLETE")
    print("=" * 55)
    print(f"✓ Successful: {successful}")
    print(f"✗ Failed:     {failed}")
    print(f"⚠ Skipped:   {skipped}")
    print(f"💾 Saved to:  {save_path}")

    return results


# ── Run batch ─────────────────────────────────────────────────────
results = process_products_batch(
    df=products_df,
    max_products=3,       # increase once confirmed working
    delay=1.5,
    save_path="product_listings.json"
)

# ── Review results ────────────────────────────────────────────────
print("\n--- RESULTS SUMMARY ---")
results_df = pd.DataFrame([{
    "index":    r["index"],
    "name":     r["name"][:40],
    "status":   r["status"],
    "title":    r.get("listing", {}).get("title", "N/A")[:40]
              if r["status"] == "success" else "N/A"
} for r in results])

print(results_df.to_string(index=False))

# ── Show one full listing ─────────────────────────────────────────
successful_results = [r for r in results if r['status'] == 'success']
if successful_results:
    print("\n--- SAMPLE FULL LISTING ---")
    sample = successful_results[0]
    listing = sample['listing']
    print(f"\nProduct:     {sample['name']}")
    print(f"Title:       {listing.get('title', 'N/A')}")
    print(f"\nDescription:\n{listing.get('description', 'N/A')}")
    print(f"\nFeatures:")
    for f in listing.get('features', []):
        print(f"  • {f}")
    print(f"\nKeywords:\n{listing.get('keywords', 'N/A')}")
    print(f"\n💾 Full results saved to: product_listings.json")

STEP 6: BATCH PROCESSING 3 PRODUCTS
Delay between calls: 1.5s
Estimated cost:      ~$0.012
Save path:           product_listings.json

[1/3] Processing: Turtle Check Men Navy Blue Shirt
----------------------------------------
  Image encoded: 2388 chars
✓ API call successful
  Tokens used — Input: 490, Output: 276, Total: 766
  ✓ Success
    Title:    Stylish Turtle Check Navy Blue Men's Shirt
    Features: 7 items
    Keywords: check shirt, navy blue shirt, men's fashion, casual shirt, s...

  Waiting 1.5s before next call...

[2/3] Processing: Peter England Men Party Blue Jeans
----------------------------------------
  Image encoded: 2292 chars
✓ API call successful
  Tokens used — Input: 490, Output: 283, Total: 773
  ✓ Success
    Title:    Stylish Peter England Blue Party Jeans for Men
    Features: 7 items
    Keywords: Peter England jeans, men's blue jeans, party jeans, casual d...

  Waiting 1.5s before next call...

[3/3] Processing: Titan Women Silver Watch
----------------

In [49]:
# ============================================================
# CLIENT REQUEST HANDLER 
# ============================================================

from pydantic import BaseModel, Field, field_validator, ValidationError
from typing import Optional, List
from datetime import datetime
import json

class ClientRequest(BaseModel):
    """
    Simulates an incoming API request from a client.
    Wraps product data with request metadata.
    """
    request_id: str = Field(
        ...,
        min_length=1,
        description="Unique request identifier"
    )
    client_name: str = Field(
        ...,
        min_length=2,
        max_length=100,
        description="Name of the client submitting the request"
    )
    products: List[dict] = Field(
        ...,
        min_length=1,
        description="List of products to process"
    )
    priority: str = Field(
        default="normal",
        description="Request priority: low, normal, high"
    )
    submitted_at: str = Field(
        default_factory=lambda: datetime.now().isoformat(),
        description="Timestamp of submission"
    )

    @field_validator('priority')
    @classmethod
    def priority_must_be_valid(cls, v):
        allowed = ['low', 'normal', 'high']
        if v not in allowed:
            raise ValueError(f"Priority must be one of {allowed}")
        return v

    @field_validator('request_id')
    @classmethod
    def request_id_strip(cls, v):
        return v.strip()


def handle_client_request(request_data: dict,
                           process_api: bool = False) -> dict:
    """
    Full client request handler:
    1. Validate the request structure
    2. Validate each product inside
    3. Optionally process via ChatGPT API
    4. Return structured response
    
    Parameters:
    - request_data: raw dict from client
    - process_api: if True, calls ChatGPT for valid products
    
    Returns:
    - response dict with status and results
    """
    print("\n" + "=" * 55)
    print(f"HANDLING CLIENT REQUEST")
    print("=" * 55)

    # ── Stage 1: Validate request structure ───────────────────────
    print("\nStage 1: Validating request structure...")
    try:
        request = ClientRequest(**request_data)
        print(f"  ✓ Request valid")
        print(f"  ID:       {request.request_id}")
        print(f"  Client:   {request.client_name}")
        print(f"  Products: {len(request.products)}")
        print(f"  Priority: {request.priority}")
    except ValidationError as e:
        errors = [f"{err['loc'][0]}: {err['msg']}" 
                  for err in e.errors()]
        print(f"  ✗ Request rejected at structure level")
        return {
            "status": "rejected",
            "stage": "request_validation",
            "errors": errors
        }

    # ── Stage 2: Validate each product ────────────────────────────
    print("\nStage 2: Validating products...")
    valid_products   = []
    invalid_products = []

    for i, product in enumerate(request.products):
        try:
            validated = ProductListingRequest(**product)
            valid_products.append(validated)
            print(f"  [{i+1}] ✓ {validated.name[:40]} | "
                  f"${validated.price}")
        except ValidationError as e:
            errors = [f"{err['loc'][0]}: {err['msg']}" 
                      for err in e.errors()]
            invalid_products.append({
                "index": i,
                "data": product,
                "errors": errors
            })
            print(f"  [{i+1}] ✗ {product.get('name', 'Unknown')[:40]}")
            for err in errors:
                print(f"       → {err}")

    print(f"\n  Valid:   {len(valid_products)}")
    print(f"  Invalid: {len(invalid_products)}")

    if not valid_products:
        return {
            "status": "rejected",
            "stage": "product_validation",
            "message": "No valid products to process",
            "invalid_products": invalid_products
        }

    # ── Stage 3: Process valid products ───────────────────────────
    processed_results = []

    if process_api:
        print("\nStage 3: Processing valid products via ChatGPT API...")
        for i, product in enumerate(valid_products):
            print(f"\n  [{i+1}/{len(valid_products)}] {product.name[:40]}")

            # Get matching image from dataset if available
            image = None
            if 'products_df' in globals():
                matching = products_df[
                    products_df.get('productDisplayName', 
                    pd.Series()).str.contains(
                        product.name[:10], 
                        case=False, 
                        na=False)]
                if len(matching) > 0:
                    image = matching.iloc[0].get('image', None)

            prompt = create_product_listing_prompt(
                product_name=product.name,
                price=product.price,
                category=product.category.value,
                additional_info=product.additional_info
            )

            raw_response = call_chatgpt_vision(image, prompt)

            if raw_response:
                parsed = parse_json_response(raw_response)
                try:
                    validated_output = GeneratedListing(**parsed)
                    processed_results.append({
                        "product": product.name,
                        "status": "success",
                        "listing": validated_output.dict()
                    })
                    print(f"  ✓ Generated: {validated_output.title[:50]}")
                except ValidationError as e:
                    processed_results.append({
                        "product": product.name,
                        "status": "output_invalid",
                        "raw": raw_response
                    })
                    print(f"  ⚠ Output validation failed")
            else:
                processed_results.append({
                    "product": product.name,
                    "status": "api_failed"
                })

            if i < len(valid_products) - 1:
                time.sleep(1.5)
    else:
        print("\nStage 3: Skipping API calls (process_api=False)")
        processed_results = [
            {"product": p.name, "status": "validated_only"}
            for p in valid_products
        ]

    # ── Stage 4: Build response ────────────────────────────────────
    response = {
        "request_id":        request.request_id,
        "client_name":       request.client_name,
        "status":            "completed",
        "summary": {
            "total_submitted": len(request.products),
            "valid":           len(valid_products),
            "invalid":         len(invalid_products),
            "processed":       len(processed_results)
        },
        "valid_products":    [p.dict() for p in valid_products],
        "invalid_products":  invalid_products,
        "results":           processed_results,
        "processed_at":      datetime.now().isoformat()
    }

    print("\n" + "=" * 55)
    print("REQUEST COMPLETE")
    print("=" * 55)
    print(f"✓ Submitted: {response['summary']['total_submitted']}")
    print(f"✓ Valid:     {response['summary']['valid']}")
    print(f"✗ Invalid:   {response['summary']['invalid']}")
    print(f"✓ Processed: {response['summary']['processed']}")

    return response


# ── Test with a realistic client request ──────────────────────────
print("=" * 55)
print("CLIENT REQUEST HANDLER TEST")
print("=" * 55)

test_request = {
    "request_id": "REQ-2026-001",
    "client_name": "FashionStore GmbH",
    "priority": "high",
    "products": [
        {
            "name": "Classic White Cotton T-Shirt",
            "price": 24.99,
            "category": "Fashion",
            "additional_info": "100% cotton, crew neck, unisex"
        },
        {
            "name": "Running Sneakers Pro",
            "price": 89.99,
            "category": "Sports",
            "additional_info": "Lightweight, breathable mesh upper"
        },
        {
            "name": "",           # invalid — empty name
            "price": 15.00,
            "category": "Fashion"
        },
        {
            "name": "Leather Belt",
            "price": -5.00,       # invalid — negative price
            "category": "Fashion"
        }
    ]
}

# Run without API calls first to test validation only
response = handle_client_request(test_request, process_api=False)

# Save response
with open("client_response.json", "w", encoding="utf-8") as f:
    json.dump(response, f, indent=2, ensure_ascii=False, default=str)
print(f"\n💾 Response saved to client_response.json")

# ── Uncomment to run WITH API calls ──────────────────────────────
# response = handle_client_request(test_request, process_api=True)

CLIENT REQUEST HANDLER TEST

HANDLING CLIENT REQUEST

Stage 1: Validating request structure...
  ✓ Request valid
  ID:       REQ-2026-001
  Client:   FashionStore GmbH
  Products: 4
  Priority: high

Stage 2: Validating products...
  [1] ✓ Classic White Cotton T-Shirt | $24.99
  [2] ✓ Running Sneakers Pro | $89.99
  [3] ✗ 
       → name: String should have at least 2 characters
  [4] ✗ Leather Belt
       → price: Input should be greater than 0

  Valid:   2
  Invalid: 2

Stage 3: Skipping API calls (process_api=False)

REQUEST COMPLETE
✓ Submitted: 4
✓ Valid:     2
✗ Invalid:   2
✓ Processed: 2

💾 Response saved to client_response.json


C:\Users\dbyst\AppData\Local\Temp\ipykernel_30488\4098393476.py:199: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  "valid_products":    [p.dict() for p in valid_products],
